In [1]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_rows', None)
import glob
import os
from tqdm import tqdm
import plotly.graph_objects as go

In [ ]:
files_1 = glob.glob(os.path.join("/Volumes/T9/CSV_GOOGL_NASDAQ_PL", "*.csv"))
files_2 = glob.glob(os.path.join("/Volumes/T9/CSV_GOOGL_NASDAQ_SL", "*.csv"))
imbalance_1 = []
imbalance_2 = []

for i in tqdm(range (len(files_1))):
    df_1 = pd.read_csv(files_1[i])
    df_2 = pd.read_csv(files_2[i])
    df_1 = df_1[df_1['side'] == 'A']
    df_1 = df_1[df_1['side'] == 'A']
    #limite = 0
    #df_1['imbalance'] = -(df_1[f'ask_sz_0{limite}']-df_1[f'bid_sz_0{limite}'])/(df_1[f'ask_sz_0{limite}']+df_1[f'bid_sz_0{limite}'])
    #limite = 1
    #df_2['imbalance'] = -(df_2[f'ask_sz_0{limite}']-df_2[f'bid_sz_0{limite}'])/(df_2[f'ask_sz_0{limite}']+df_2[f'bid_sz_0{limite}'])
    df_1['imbalance'] = df_1['imbalance'].shift()
    #df_1['Mean_price_diff'] = df_1['price'].shift(-50) - df_1['price']
    df_1 = df_1.dropna()
    df_1 = df_1[:-100]
    df_2['imbalance'] = df_2['imbalance'].shift()
    #df_2['Mean_price_diff'] = df_2['price'].shift(-50) - df_2['price']
    df_2 = df_2.dropna()
    df_2 = df_2[:-100]
    df_1 = df_1[['ts_event',  'imbalance']]
    df_2 = df_2[['ts_event',  'imbalance']]
    df_1['ts_event'] = pd.to_datetime(df_1['ts_event'])
    df_2['ts_event'] = pd.to_datetime(df_2['ts_event'])
    df_1 = df_1[(df_1['ts_event'].dt.hour > 14) | ((df_1['ts_event'].dt.hour == 14) & (df_1['ts_event'].dt.minute >= 30)) & (df_1['ts_event'].dt.hour < 19)]
    df_1 = df_2[(df_2['ts_event'].dt.hour > 14) | ((df_2['ts_event'].dt.hour == 14) & (df_2['ts_event'].dt.minute >= 30)) & (df_2['ts_event'].dt.hour < 19)]
    df1_grouped = df_1.resample('5s', on='ts_event').mean().reset_index()
    df2_grouped = df_2.resample('5s', on='ts_event').mean().reset_index()
    df2_grouped = df2_grouped.dropna()
    df1_grouped = df1_grouped.dropna()
    imbalance_1.append(df1_grouped['imbalance'].to_numpy())
    imbalance_2.append(df2_grouped['imbalance'].to_numpy())
    
imbalance_1 = np.concatenate(imbalance_1)
imbalance_2 = np.concatenate(imbalance_2)
indices = np.argsort(imbalance_1)
im_1 = imbalance_1[indices]
im_2 = imbalance_2[indices]

In [ ]:
group_size = 10000

im_1_group = [im_1[i:i + group_size] for i in range(0, len(im_1), group_size)]
im_2_group = [im_2[i:i + group_size] for i in range(0, len(im_2), group_size)]

im_11 = np.array([np.mean(group) for group in im_1_group])
im_22 = np.array([np.mean(group) for group in im_2_group])

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=im_11-np.mean(im_11),  # Imbalance de df1
    y=im_22-np.mean(im_22),  # Imbalance de df2
    mode='markers+lines',
    name='Imbalance Relationship',
    marker=dict(color='blue', size=8),
    line=dict(color='blue', width=2)
))

# Mise à jour de la mise en page
fig.update_layout(
    title='Imbalance de DF1 en fonction de DF2',
    xaxis_title='Imbalance DF1',
    yaxis_title='Imbalance DF2',
    plot_bgcolor='white',
    width=800,
    height=500
)

# Afficher le graphique
fig.show()

In [ ]:
np.mean(im_1)

In [ ]:
df = pd.read_csv('/Volumes/T9/CSV_dezippe_nasdaq/xnas-itch-20240724.mbp-10.csv')
df = df[df['symbol'] == 'GOOGL']
df.head()

In [ ]:
df_trades = df[df['action'] == 'T']
prices = df_trades['price'].to_numpy()
time_trades = pd.to_datetime(df_trades['ts_event'])
fig = go.Figure()
fig.add_trace(go.Scatter(x = time_trades, y = prices, mode ='lines', name ='Prix', line=dict(color = 'red')))
fig.update_layout(title=f'Évolution du prix de GOOGL avec le bid-ask entre 16h et 16h05', xaxis_title='Temps', yaxis_title='Prix', showlegend=False)
fig.show()


In [ ]:
from plotly.colors import sample_colorscale

df['ts_event'] = pd.to_datetime(df['ts_event'])
# df = df[
#     ((df['ts_event'].dt.hour == 17) & (df['ts_event'].dt.minute == 59)) |
#     ((df['ts_event'].dt.hour == 18) & (df['ts_event'].dt.minute >= 0) & (df['ts_event'].dt.minute <= 3))
# ]

fig = go.Figure()
time = pd.to_datetime(df['ts_event'].to_numpy())

red_shades = sample_colorscale('Reds', [i / 9 for i in range(10)])

# for i in range(10):  # Pour px_00 à px_09
#     bid = df[f'bid_px_0{i}'].to_numpy()
#     ask = df[f'ask_px_0{i}'].to_numpy()
#     color = red_shades[i]  # Choix de la couleur
#     fig.add_trace(go.Scatter(x=time, y=bid, mode='lines', name=f'Bid px_0{i}', line=dict(color=color)))
#     fig.add_trace(go.Scatter(x=time, y=ask, mode='lines', name=f'Ask px_0{i}', line=dict(color=color)))
df_trades = df[df['action'] == 'T']
prices = df_trades['price'].to_numpy()
time_trades = pd.to_datetime(df_trades['ts_event'])
fig.add_trace(go.Scatter(x = time_trades, y = prices, mode ='lines', name ='Prix', line=dict(color = 'red')))
fig.update_layout(title=f'', xaxis_title='Temps', yaxis_title='Prix', showlegend=False)


fig.show()

In [ ]:
df.head(1)



In [ ]:

for i in range(10):  # Pour px_00 à px_09
    print(np.mean(df[f'bid_sz_0{i}'].to_numpy()), i)
    print(np.mean(df[f'ask_sz_0{i}'].to_numpy()))

In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.colors import sample_colorscale

# Filtrage des données entre 16h00 et 16h05
df['ts_event'] = pd.to_datetime(df['ts_event'])
df = df[
    (df['ts_event'].dt.hour == 1) &
    (df['ts_event'].dt.minute >= 2) &
    (df['ts_event'].dt.minute < 3)
]

# Initialisation de la figure
fig = go.Figure()
time = pd.to_datetime(df['ts_event'].to_numpy())

# Génération des couleurs rouges avec Plotly
red_shades = sample_colorscale('RdBu', [ (i / 9) for i in range(10)])


# Ajout des courbes Bid et Ask avec un style épuré
for i in range(10):  # Pour px_00 à px_09
    bid = df[f'bid_px_0{i}'].to_numpy()
    ask = df[f'ask_px_0{i}'].to_numpy()
    color = red_shades[i]
    fig.add_trace(go.Scatter(
        x=time, y=bid, mode='lines', name=f'Bid px_0{i}', 
        line=dict(color=color, width=1.2, dash='solid')  # Ligne fine et continue
    ))
    fig.add_trace(go.Scatter(
        x=time, y=ask, mode='lines', name=f'Ask px_0{i}', 
        line=dict(color=color, width=1.2, dash='solid')  # Ligne fine et continue
    ))

# Ajout des prix des trades avec une couleur distinctive
df_trades = df[df['action'] == 'T']
prices = df_trades['price'].to_numpy()
time_trades = pd.to_datetime(df_trades['ts_event'])
fig.add_trace(go.Scatter(
    x=time_trades, y=prices, mode='markers', name='Trades',
    marker=dict(color='black', size=6, symbol='circle')  # Points noirs
))

# Mise à jour du style de la figure
fig.update_layout(
    #title='Évolution du prix de GOOGL avec le bid-ask entre 16h et 16h05',
    #title_font=dict(size=16, family='Times New Roman'),  # Police LaTeX-like
   # xaxis=dict(
    #    title='Temps',
    #    title_font=dict(size=14, family='Times New Roman'),
    #    tickfont=dict(size=12, family='Times New Roman'),
    #    showgrid=True, gridwidth=0.5, gridcolor='lightgrey',  # Grille légère
    #    zeroline=False, linecolor='black'
    #),
    yaxis=dict(
        title='Prix',
        title_font=dict(size=14, family='Times New Roman'),
        tickfont=dict(size=12, family='Times New Roman'),
        showgrid=True, gridwidth=0.5, gridcolor='lightgrey',  # Grille légère
        zeroline=False, linecolor='black'
    ),
    plot_bgcolor='white',  # Fond blanc
    showlegend=False,
    legend=dict(
        font=dict(size=10, family='Times New Roman'),
        bordercolor='black', borderwidth=0.5
    ),
    width=1000,  # Largeur du graphe en pixels
    height=600 
)

# Affichage du graphique
fig.show()



In [ ]:
# print(len(df))

In [ ]:
files_csv = glob.glob(os.path.join("/Volumes/T9/CSV_dezippe_nasdaq", "*.csv"))
print(len(files_csv))
Add_GOOGL = 0
Add_KHC = 0
Add_LCID = 0
T_GOOGL = 0
T_LCID = 0
T_KHC = 0
Cancel_GOOGL = 0
Cancel_LCID = 0
Cancel_KHC = 0
Total_GOOGL = 0
Total_KHC = 0
Total_LCID = 0
for f in tqdm(files_csv):
    df = pd.read_csv(f)
    df_G = df[df['symbol'] == 'GOOGL']
    df_KHC = df[df['symbol'] == 'KHC']
    df_LCID = df[df['symbol'] == 'LCID']
    Add_GOOGL += len(df_G[df_G['action'] == 'A'])
    Add_KHC += len(df_KHC[df_KHC['action'] == 'A'])
    Add_LCID += len(df_LCID[df_LCID['action'] == 'A'])
    T_GOOGL += len(df_G[df_G['action'] == 'T'])
    T_LCID += len(df_LCID[df_LCID['action'] == 'T'])
    T_KHC += len(df_KHC[df_KHC['action'] == 'T'])
    Cancel_GOOGL += len(df_G[df_G['action'] == 'C'])
    Cancel_LCID += len(df_LCID[df_LCID['action'] == 'C'])
    Cancel_KHC += len(df_KHC[df_KHC['action'] == 'C'])
    Total_GOOGL += len(df_G)
    Total_LCID += len(df_LCID)
    Total_KHC += len(df_KHC)

In [ ]:
print(Add_GOOGL)
print(Cancel_GOOGL)
print(T_GOOGL)
print(Total_GOOGL)

In [ ]:
print(Add_LCID)
print(Cancel_LCID)
print(T_LCID)
print(Total_LCID)

In [ ]:
print(Add_KHC)
print(Cancel_KHC)
print(T_KHC)
print(Total_KHC)

In [ ]:
print(Add_LCID/Total_LCID)
print(Cancel_LCID/Total_LCID)
print(T_LCID/Total_LCID)
print(Total_LCID/65)